# 🧩 The Complete Guide to Merging, Joining & Concatenating DataFrames in Pandas

---

## 📋 Table of Contents

1. [Setup & Data Loading](#1-setup)
2. [Understanding the Datasets](#2-datasets)
3. [pd.merge() — The Core Function](#3-merge)
   - 3.1 [Inner Join — `how='inner'`](#3-1-inner)
   - 3.2 [Left Join — `how='left'`](#3-2-left)
   - 3.3 [Right Join — `how='right'`](#3-3-right)
   - 3.4 [Outer (Full) Join — `how='outer'`](#3-4-outer)
   - 3.5 [Cross Join — `how='cross'`](#3-5-cross)
4. [Merging on Different Key Configurations](#4-keys)
   - 4.1 [Merging on a Single Key — `on`](#4-1-on)
   - 4.2 [Merging on Multiple Keys — `on=[...]`](#4-2-multiple-keys)
   - 4.3 [Merging on Different Column Names — `left_on` / `right_on`](#4-3-left-right-on)
   - 4.4 [Merging on the Index — `left_index` / `right_index`](#4-4-index)
   - 4.5 [Mixing Index and Column — `left_index` + `right_on`](#4-5-index-column)
5. [Handling Duplicate & Suffix Columns — `suffixes`](#5-suffixes)
6. [Controlling Extra Columns — `indicator`](#6-indicator)
7. [Validation — `validate`](#7-validate)
8. [DataFrame.join() — Index-Based Joining](#8-join)
   - 8.1 [Basic join()](#8-1-basic-join)
   - 8.2 [join() with `on` parameter](#8-2-join-on)
   - 8.3 [join() with `how` parameter](#8-3-join-how)
   - 8.4 [Joining Multiple DataFrames at Once](#8-4-multi-join)
9. [pd.concat() — Stacking DataFrames](#9-concat)
   - 9.1 [Vertical (Row) Concatenation — `axis=0`](#9-1-vertical)
   - 9.2 [Horizontal (Column) Concatenation — `axis=1`](#9-2-horizontal)
   - 9.3 [Handling Indexes with `ignore_index`](#9-3-ignore-index)
   - 9.4 [Creating Hierarchical Index with `keys`](#9-4-keys)
   - 9.5 [Controlling Missing Values with `join` in concat](#9-5-concat-join)
   - 9.6 [Verifying No Duplicates with `verify_integrity`](#9-6-verify)
10. [DataFrame.append() — Legacy Row Addition (Deprecated)](#10-append)
11. [Advanced Real-World Scenarios](#11-advanced)
    - 11.1 [Multi-Table Star Schema Join (Instacart)](#11-1-star)
    - 11.2 [Self-Join — Comparing Rows Within the Same Table](#11-2-self-join)
    - 11.3 [Merging with Aggregated Data](#11-3-agg-merge)
    - 11.4 [Fuzzy/Nearest Merge — `merge_asof`](#11-4-merge-asof)
    - 11.5 [Ordered Merge — `merge_ordered`](#11-5-merge-ordered)
    - 11.6 [Chaining Multiple Merges — Pipeline Style](#11-6-chain)
12. [Performance Tips & Best Practices](#12-performance)
13. [Quick Reference Cheat Sheet](#13-cheatsheet)

---

> **How to use this notebook:** Every section is driven by a **practical question** — the best way to learn is to understand *why* you need a particular technique before learning *how* to use it.

---
## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# Pretty display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 120)

print(f"pandas version: {pd.__version__}")
print(f"numpy  version: {np.__version__}")

### 1.1 Loading the Instacart Dataset

We use the Instacart grocery-orders dataset which contains five related tables that form a classic **star schema** — a central fact table (`order_products`) surrounded by dimension tables. This structure makes it perfect for practising joins.

Update the `BASE_PATH` below to match your environment.

In [ ]:
# ── Adjust this path to match your own Google Drive / local folder ───────────
BASE_PATH = ''

# Load all five Instacart tables (semicolon-separated CSVs)
order_products   = pd.read_csv(BASE_PATH + 'order_products.csv',   sep=';')
aisles           = pd.read_csv(BASE_PATH + 'aisles.csv',           sep=';')
departments      = pd.read_csv(BASE_PATH + 'departments.csv',      sep=';')
instacart_orders = pd.read_csv(BASE_PATH + 'instacart_orders.csv', sep=';')
products         = pd.read_csv(BASE_PATH + 'products.csv',         sep=';')

print("Tables loaded successfully!")
print(f"  order_products   : {order_products.shape}")
print(f"  aisles           : {aisles.shape}")
print(f"  departments      : {departments.shape}")
print(f"  instacart_orders : {instacart_orders.shape}")
print(f"  products         : {products.shape}")

### 1.2 Synthetic Toy Datasets

For many merge examples we need **full control over the data** so we can clearly see what happens to matching and non-matching rows. We create small, self-explanatory toy tables here and reuse them throughout the notebook.

In [ ]:
# ── Toy dataset A: Employees ─────────────────────────────────────────────────
employees = pd.DataFrame({
    'emp_id':   [1, 2, 3, 4, 5],
    'name':     ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept_id':  [10, 20, 10, 30, 99],     # 99 → department does NOT exist
    'salary':   [70000, 85000, 72000, 90000, 65000]
})

# ── Toy dataset B: Departments ───────────────────────────────────────────────
departments_toy = pd.DataFrame({
    'dept_id':   [10, 20, 30, 40],        # 40 → no employee belongs here
    'dept_name': ['Engineering', 'Marketing', 'Finance', 'HR'],
    'budget':    [500000, 300000, 400000, 200000]
})

# ── Toy dataset C: Sales ─────────────────────────────────────────────────────
sales = pd.DataFrame({
    'emp_id':  [1, 1, 2, 3, 3, 5],
    'quarter': ['Q1', 'Q2', 'Q1', 'Q1', 'Q2', 'Q1'],
    'revenue': [10000, 12000, 8000, 15000, 14000, 5000]
})

# ── Toy dataset D: Bonuses (duplicate emp_ids on right side) ─────────────────
bonuses = pd.DataFrame({
    'emp_id': [1, 1, 2, 3],              # emp_id=1 gets TWO bonus rows
    'bonus':  [500, 300, 700, 200]
})

print("employees\n", employees, "\n")
print("departments_toy\n", departments_toy, "\n")
print("sales\n", sales)

---
## 2. Understanding the Datasets <a id='2-datasets'></a>

Before merging, always understand the **entity-relationship** between your tables.

### ❓ Question: What are the relationships between the Instacart tables, and which columns link them together?

```
instacart_orders ──── order_id ────► order_products
products         ──── product_id ──► order_products
products         ──── aisle_id ────► aisles
products         ──── department_id ► departments
```

In [ ]:
# ── Quick schema check: preview each table and its key columns ────────────────
for name, df in [
    ('instacart_orders', instacart_orders),
    ('order_products',   order_products),
    ('products',         products),
    ('aisles',           aisles),
    ('departments',      departments)
]:
    print(f"\n{'─'*60}")
    print(f"TABLE: {name}  {df.shape}")
    print(f"Columns: {list(df.columns)}")
    display(df.head(2))

---
## 3. pd.merge() — The Core Function <a id='3-merge'></a>

`pd.merge()` is the Swiss-Army knife of pandas data combination. Its signature:

```python
pd.merge(
    left, right,
    how        = 'inner',     # join type
    on         = None,        # shared column name(s)
    left_on    = None,        # left key column(s)
    right_on   = None,        # right key column(s)
    left_index = False,       # use left index as key
    right_index= False,       # use right index as key
    suffixes   = ('_x','_y'), # suffix for overlapping columns
    indicator  = False,       # add _merge column
    validate   = None,        # check cardinality
    sort       = False        # sort result keys
)
```

The diagram below summarises the four fundamental join types:

```
LEFT   RIGHT     INNER           LEFT            RIGHT           OUTER
A  B   B  C      A  B  C         A  B  C         A  B  C         A  B  C
─────  ─────    ──────────      ──────────────  ──────────────  ──────────────
1  a   a  X     1  a  X         1  a  X         1  a  X         1  a  X
2  b   c  Y     3  c  Y         2  b  NaN       3  c  Y         2  b  NaN
3  c              (matched       3  c  Y         NaN d  Z        3  c  Y
                  only)          (all left)      (all right)     4  d  Z
                                                                (all rows)
```

### 3.1 Inner Join — `how='inner'` <a id='3-1-inner'></a>

#### ❓ Question: Which employees actually belong to a known department? (Show only matched rows from both sides.)

An **inner join** keeps only rows where the key exists in **both** tables. Rows with no match are dropped from the result.

In [ ]:
# Inner join: only employees whose dept_id exists in departments_toy
inner = pd.merge(
    left  = employees,
    right = departments_toy,
    on    = 'dept_id',    # shared key column
    how   = 'inner'       # DEFAULT — could omit this
)

print(f"employees rows: {len(employees)}  →  inner join rows: {len(inner)}")
print("\n⚠️  Eve (dept_id=99) was DROPPED — 99 doesn't exist in departments_toy")
print("⚠️  HR dept (dept_id=40) was DROPPED — no employee belongs there\n")
inner

### 3.2 Left Join — `how='left'` <a id='3-2-left'></a>

#### ❓ Question: Show ALL employees, and add department info where it exists. Flag employees with no matching department.

A **left join** keeps every row from the **left** DataFrame, filling right-side columns with `NaN` when there is no match.

In [ ]:
# Left join: preserve all employees, attach dept info where available
left_join = pd.merge(
    left  = employees,
    right = departments_toy,
    on    = 'dept_id',
    how   = 'left'
)

print(f"employees rows: {len(employees)}  →  left join rows: {len(left_join)}")
print("\n✅ Eve is retained with NaN dept_name and NaN budget\n")
left_join

In [ ]:
# Practical use: find employees with NO matching department (unmatched rows)
unmatched = left_join[left_join['dept_name'].isna()]
print("Employees with unknown department:")
unmatched[['emp_id', 'name', 'dept_id', 'dept_name']]

### 3.3 Right Join — `how='right'` <a id='3-3-right'></a>

#### ❓ Question: Show ALL departments — even those with no employees — alongside any employees who belong to them.

A **right join** keeps every row from the **right** DataFrame. It is the mirror image of a left join.

In [ ]:
# Right join: preserve all departments, attach employee info where available
right_join = pd.merge(
    left  = employees,
    right = departments_toy,
    on    = 'dept_id',
    how   = 'right'
)

print(f"Result rows: {len(right_join)}")
print("\n✅ HR (dept_id=40) is retained with NaN employee fields\n")
right_join

In [ ]:
# 💡 Tip: a right join is always equivalent to swapping left/right and using left
equivalent = pd.merge(departments_toy, employees, on='dept_id', how='left')
print("Right join == swapped left join? ",
      set(right_join.columns) == set(equivalent.columns))  # same columns, different order

### 3.4 Outer (Full) Join — `how='outer'` <a id='3-4-outer'></a>

#### ❓ Question: Give me a complete picture — every employee AND every department, matched where possible, with NaN for any gaps.

An **outer join** (also called a *full outer join*) is the union of left and right joins. No row from either side is discarded.

In [ ]:
# Outer join: keep ALL rows from both sides
outer_join = pd.merge(
    left  = employees,
    right = departments_toy,
    on    = 'dept_id',
    how   = 'outer'
)

print(f"Left rows: {len(employees)} | Right rows: {len(departments_toy)} | Outer rows: {len(outer_join)}")
print("\n✅ Eve (unmatched left) + HR (unmatched right) are BOTH retained\n")
outer_join

### 3.5 Cross Join — `how='cross'` <a id='3-5-cross'></a>

#### ❓ Question: I want every possible combination of employees and quarters (e.g. to build a salary planning template). How do I get a Cartesian product?

A **cross join** creates every pair of rows from left × right. No key column is needed. The result has `len(left) × len(right)` rows.

In [ ]:
# Small tables for cross join demo
employees_small = employees[['emp_id', 'name']].head(3).copy()

quarters = pd.DataFrame({'quarter': ['Q1', 'Q2', 'Q3', 'Q4']})

# Cross join: every employee × every quarter
cross = pd.merge(
    left  = employees_small,
    right = quarters,
    how   = 'cross'          # no 'on' parameter — not needed
)

print(f"3 employees × 4 quarters = {len(cross)} rows")
cross

---
## 4. Merging on Different Key Configurations <a id='4-keys'></a>

### 4.1 Merging on a Single Key — `on` <a id='4-1-on'></a>

#### ❓ Question: How do I join orders with products when both tables share a column called `product_id`?

In [ ]:
# When both DataFrames have the SAME column name, use `on`
orders_with_products = pd.merge(
    left  = order_products,
    right = products,
    on    = 'product_id',   # column name exists in BOTH tables
    how   = 'inner'
)

print(f"order_products: {order_products.shape}")
print(f"products:       {products.shape}")
print(f"merged result:  {orders_with_products.shape}\n")
orders_with_products.head(3)

### 4.2 Merging on Multiple Keys — `on=[...]` <a id='4-2-multiple-keys'></a>

#### ❓ Question: Sales data has both `emp_id` AND `quarter`. How do I merge it with a targets table that also uses both columns as a composite key?

In [ ]:
# Create a targets table with a composite key (emp_id + quarter)
targets = pd.DataFrame({
    'emp_id':  [1, 1, 2, 3, 3],
    'quarter': ['Q1', 'Q2', 'Q1', 'Q1', 'Q2'],
    'target':  [9000, 11000, 7500, 14000, 13000]
})

# Merge on TWO columns simultaneously — both must match
sales_vs_target = pd.merge(
    left  = sales,
    right = targets,
    on    = ['emp_id', 'quarter'],   # composite key — list of columns
    how   = 'left'
)

# Compute performance vs target
sales_vs_target['vs_target'] = sales_vs_target['revenue'] - sales_vs_target['target']
sales_vs_target

### 4.3 Merging on Different Column Names — `left_on` / `right_on` <a id='4-3-left-right-on'></a>

#### ❓ Question: The orders table calls it `order_id` but the shipments table calls it `shipment_order_id`. How do I merge tables whose key columns have different names?

In [ ]:
# Simulate mismatched column names
shipments = pd.DataFrame({
    'shipment_order_id': [2141543, 567889, 9999999],  # 9999999 won't match
    'carrier':           ['FedEx', 'UPS', 'DHL'],
    'shipped_date':      ['2024-01-10', '2024-01-11', '2024-01-12']
})

# left_on  → column name in the LEFT  DataFrame
# right_on → column name in the RIGHT DataFrame
orders_shipped = pd.merge(
    left     = order_products.head(5),
    right    = shipments,
    left_on  = 'order_id',           # name in order_products
    right_on = 'shipment_order_id',  # name in shipments
    how      = 'left'
)

# Note: BOTH key columns are kept; drop the redundant one
orders_shipped = orders_shipped.drop(columns='shipment_order_id')
orders_shipped

### 4.4 Merging on the Index — `left_index` / `right_index` <a id='4-4-index'></a>

#### ❓ Question: My lookup table uses the index as its key (e.g., `aisle_id` is the index). How do I merge without resetting the index first?

In [ ]:
# Set aisle_id as the index in aisles
aisles_indexed = aisles.set_index('aisle_id')
print("aisles_indexed (first 3 rows):")
display(aisles_indexed.head(3))

# Merge products (has aisle_id as a column) with aisles_indexed (has it as index)
products_with_aisle = pd.merge(
    left        = products,
    right       = aisles_indexed,
    left_on     = 'aisle_id',   # column in products
    right_index = True,         # use the index of aisles_indexed
    how         = 'left'
)

print(f"\nResult shape: {products_with_aisle.shape}")
products_with_aisle.head(4)

### 4.5 Mixing Index and Column — `left_index` + `right_on` <a id='4-5-index-column'></a>

#### ❓ Question: What if my LEFT table uses the index as the key, and the RIGHT table uses a regular column?

In [ ]:
# Set product_id as the index on the products side
products_indexed = products.set_index('product_id')

# order_products has product_id as a regular column
merged_idx_col = pd.merge(
    left         = products_indexed,
    right        = order_products.head(10),
    left_index   = True,          # use index of products_indexed
    right_on     = 'product_id',  # column in order_products
    how          = 'inner'
)

merged_idx_col.head(5)

---
## 5. Handling Duplicate & Suffix Columns — `suffixes` <a id='5-suffixes'></a>

#### ❓ Question: Both tables have a column named `aisle_id`. After merging, how do I tell them apart, and how do I rename them to something meaningful?

In [ ]:
# Scenario: two tables that BOTH have a column called 'rating'
product_ratings = pd.DataFrame({
    'product_id': [1, 2, 3, 4],
    'rating':     [4.5, 3.8, 4.9, 2.1]   # customer rating
})

editor_ratings = pd.DataFrame({
    'product_id': [1, 2, 3, 5],
    'rating':     [4.0, 4.2, 4.7, 3.5]   # editorial rating
})

# DEFAULT suffixes: _x for left, _y for right
default_suffix = pd.merge(product_ratings, editor_ratings, on='product_id', how='outer')
print("Default suffixes (_x, _y):")
display(default_suffix)

# CUSTOM suffixes: descriptive names
custom_suffix = pd.merge(
    product_ratings, editor_ratings,
    on       = 'product_id',
    how      = 'outer',
    suffixes = ('_customer', '_editor')   # tuple: (left_suffix, right_suffix)
)
print("\nCustom suffixes (_customer, _editor):")
display(custom_suffix)

---
## 6. Controlling Extra Columns — `indicator` <a id='6-indicator'></a>

#### ❓ Question: After an outer merge, I want a column that tells me whether each row came from the left table only, the right table only, or both. How?

Setting `indicator=True` adds a `_merge` column with values: `'left_only'`, `'right_only'`, or `'both'`.

In [ ]:
# indicator=True adds a '_merge' column explaining where each row came from
indicated = pd.merge(
    left      = employees,
    right     = departments_toy,
    on        = 'dept_id',
    how       = 'outer',
    indicator = True       # adds '_merge' column
)
print(indicated[['emp_id', 'name', 'dept_id', 'dept_name', '_merge']])

In [ ]:
# Use indicator to find rows that exist in LEFT but NOT in RIGHT (anti-join)
left_anti = indicated[indicated['_merge'] == 'left_only']
print("Anti-join (employees with no matching department):")
display(left_anti[['emp_id', 'name', 'dept_id']])

# Custom name for the indicator column
indicated2 = pd.merge(
    employees, departments_toy, on='dept_id', how='outer',
    indicator='source'   # string → column gets that name
)
print("\nCustom indicator column named 'source':")
display(indicated2[['emp_id', 'dept_id', 'source']])

---
## 7. Validation — `validate` <a id='7-validate'></a>

#### ❓ Question: I expect every `order_id` in `order_products` to match exactly ONE row in `instacart_orders`. How do I enforce that assumption and catch duplicates or mismatches early?

The `validate` parameter checks the cardinality (relationship type) of the merge and raises a `MergeError` if the assumption is violated.

| Value | Meaning |
|---|---|
| `'1:1'` | Each key appears once in both tables |
| `'1:m'` | Each left key appears once; right key may repeat |
| `'m:1'` | Left key may repeat; each right key appears once |
| `'m:m'` | Both keys may repeat (no check, but documents intent) |

In [ ]:
# ── VALID case: many order_products rows per order_id (m:1 vs instacart_orders) ─
valid_merge = pd.merge(
    left     = order_products.head(100),
    right    = instacart_orders,
    on       = 'order_id',
    how      = 'inner',
    validate = 'm:1'    # many order_product rows → one order row
)
print(f"Valid m:1 merge succeeded! Shape: {valid_merge.shape}")

In [ ]:
# ── INVALID case: claiming 1:1 when bonuses has duplicate emp_ids ─────────────
try:
    bad_merge = pd.merge(
        left     = employees,
        right    = bonuses,
        on       = 'emp_id',
        how      = 'left',
        validate = '1:1'   # WRONG — bonuses has two rows for emp_id=1
    )
except pd.errors.MergeError as e:
    print(f"MergeError caught: {e}")
    print("\n✅ validate caught the problem before it caused silent data duplication!")

In [ ]:
# ── Correct cardinality for employees ↔ bonuses is 1:m ───────────────────────
correct_merge = pd.merge(
    employees, bonuses, on='emp_id', how='left', validate='1:m'
)
print("1:m merge succeeded — notice emp_id=1 now has two rows:")
correct_merge[correct_merge['emp_id'] == 1]

---
## 8. DataFrame.join() — Index-Based Joining <a id='8-join'></a>

`DataFrame.join()` is a convenience wrapper around `pd.merge()` that defaults to **index-on-index** joining. It's cleaner to write when your keys live in the index.

```python
df.join(
    other,
    on     = None,     # use this column of SELF to join on other's INDEX
    how    = 'left',   # default is LEFT (unlike merge's default of INNER)
    lsuffix= '',
    rsuffix= ''
)
```

> ⚠️ **Key difference from merge:** `join()` defaults to `how='left'`, while `merge()` defaults to `how='inner'`.

### 8.1 Basic join() — Index to Index <a id='8-1-basic-join'></a>

#### ❓ Question: Both my DataFrames are indexed by `emp_id`. What's the simplest way to combine them?

In [ ]:
# Set emp_id as index on both sides
emp_indexed = employees.set_index('emp_id')[['name', 'salary']]

dept_lookup = employees.set_index('emp_id')[['dept_id']].merge(
    departments_toy, on='dept_id'
).set_index(
    employees['emp_id'].values  # reconstruct emp_id index
)[['dept_name']]

# join() automatically aligns on the shared index
result = emp_indexed.join(dept_lookup, how='left')
print("Index-to-index join:")
result

### 8.2 join() with `on` parameter <a id='8-2-join-on'></a>

#### ❓ Question: My left table has `dept_id` as a regular column, but the right lookup table has `dept_id` as its index. Can I use join() without resetting the index?

In [ ]:
# Right table: dept info indexed by dept_id
dept_indexed = departments_toy.set_index('dept_id')
print("dept_indexed:")
display(dept_indexed)

# Left table: employees with dept_id as a COLUMN
# on='dept_id' → use the dept_id COLUMN of employees to look up dept_indexed's INDEX
joined = employees.join(
    dept_indexed,
    on  = 'dept_id',   # column in SELF to match against other's index
    how = 'left'
)
print("\njoin(on='dept_id') result:")
joined

### 8.3 join() with `how` parameter <a id='8-3-join-how'></a>

#### ❓ Question: How do the four join types (`inner`, `left`, `right`, `outer`) behave with join()?

In [ ]:
# Two indexed tables with partial overlap
df_a = pd.DataFrame({'A': ['a1', 'a2', 'a3']}, index=[1, 2, 3])
df_b = pd.DataFrame({'B': ['b2', 'b3', 'b4']}, index=[2, 3, 4])

for how in ['left', 'right', 'inner', 'outer']:
    result = df_a.join(df_b, how=how)
    print(f"how='{how}' → rows={len(result)}")
    print(result, "\n")

### 8.4 Joining Multiple DataFrames at Once <a id='8-4-multi-join'></a>

#### ❓ Question: I have five separate tables all indexed by the same key. Do I have to merge them one at a time, or can I join them all in a single call?

In [ ]:
# join() accepts a LIST of DataFrames — joins them all at once
base        = employees.set_index('emp_id')[['name']]
salaries_df = employees.set_index('emp_id')[['salary']]
dept_df     = employees.set_index('emp_id')[['dept_id']]

# Single call, three tables
combined = base.join([salaries_df, dept_df], how='left')
print("Multi-table join result:")
combined

---
## 9. pd.concat() — Stacking DataFrames <a id='9-concat'></a>

`pd.concat()` is for **combining** DataFrames that have the same structure, rather than relating them by a key.

```python
pd.concat(
    objs,                   # list (or dict) of DataFrames
    axis             = 0,   # 0 = stack rows, 1 = stack columns
    join             = 'outer',
    ignore_index     = False,
    keys             = None, # create hierarchical index
    verify_integrity = False,
    sort             = False
)
```

### 9.1 Vertical (Row) Concatenation — `axis=0` <a id='9-1-vertical'></a>

#### ❓ Question: I receive monthly order files separately. How do I stack January, February, and March orders into a single DataFrame?

In [ ]:
# Simulate three monthly slices
jan = instacart_orders.sample(5, random_state=1).copy()
feb = instacart_orders.sample(5, random_state=2).copy()
mar = instacart_orders.sample(5, random_state=3).copy()

jan['month'] = 'January'
feb['month'] = 'February'
mar['month'] = 'March'

# Stack them vertically (default axis=0)
all_months = pd.concat([jan, feb, mar], axis=0)

print(f"jan:{len(jan)} + feb:{len(feb)} + mar:{len(mar)} = {len(all_months)} rows")
print(f"Index values: {list(all_months.index)}")
print("\n⚠️  Index values are REPEATED — they come from the original DataFrames")
all_months.head(6)

### 9.2 Horizontal (Column) Concatenation — `axis=1` <a id='9-2-horizontal'></a>

#### ❓ Question: I have two DataFrames with different columns for the same set of rows. How do I place them side-by-side?

In [ ]:
# Demographics table (same 5 rows)
demographics = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5],
    'age':    [32, 45, 28, 51, 36],
    'city':   ['London', 'Berlin', 'Paris', 'Tokyo', 'Sydney']
})

performance = pd.DataFrame({
    'emp_id':  [1, 2, 3, 4, 5],
    'score':   [92, 78, 85, 90, 70],
    'rank':    ['A', 'C', 'B', 'A', 'C']
})

# axis=1 stacks columns side by side (aligns on row index)
combined_cols = pd.concat([demographics, performance], axis=1)
print("Horizontal concat (axis=1):")
combined_cols

### 9.3 Handling Indexes with `ignore_index` <a id='9-3-ignore-index'></a>

#### ❓ Question: After concatenating, my index has duplicate values like 0,1,2,0,1,2. How do I get a clean sequential index?

In [ ]:
# Without ignore_index: original indexes are preserved (may duplicate)
without_reset = pd.concat([jan, feb], axis=0)
print("Without ignore_index — index:", list(without_reset.index)[:8])

# With ignore_index=True: new sequential index 0, 1, 2, ...
with_reset = pd.concat([jan, feb], axis=0, ignore_index=True)
print("With ignore_index=True  — index:", list(with_reset.index))

### 9.4 Creating Hierarchical Index with `keys` <a id='9-4-keys'></a>

#### ❓ Question: I want to concatenate monthly data but still be able to slice by month later. How do I label each chunk?

In [ ]:
# keys adds an outer level to the MultiIndex — one label per DataFrame
hierarchical = pd.concat(
    [jan, feb, mar],
    keys = ['January', 'February', 'March'],   # label for each chunk
    axis = 0
)

print("Shape:", hierarchical.shape)
print("Index type:", type(hierarchical.index))
print()
display(hierarchical.head(3))

# Slice just February using the outer level
print("\nSlicing only February rows:")
display(hierarchical.loc['February'].head(3))

In [ ]:
# keys also work with a DICT — keys become the outer level automatically
monthly_dict = {'January': jan, 'February': feb, 'March': mar}
from_dict = pd.concat(monthly_dict)
print("From dict — same result:")
from_dict.head(3)

### 9.5 Controlling Missing Values with `join` in concat <a id='9-5-concat-join'></a>

#### ❓ Question: When I concat two DataFrames with different columns, what happens to the missing cells? Can I drop columns that don't appear in all DataFrames?

In [ ]:
# Two DataFrames with PARTIALLY overlapping columns
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df2 = pd.DataFrame({'B': [5, 6], 'C': [7, 8]})  # has B and C, but not A

# Default join='outer': keeps ALL columns, fills gaps with NaN
outer_concat = pd.concat([df1, df2], axis=0, ignore_index=True)
print("join='outer' (default) — keeps all columns:")
display(outer_concat)

# join='inner': keeps ONLY columns that appear in EVERY DataFrame
inner_concat = pd.concat([df1, df2], axis=0, join='inner', ignore_index=True)
print("join='inner' — keeps only shared columns:")
display(inner_concat)

### 9.6 Verifying No Duplicates with `verify_integrity` <a id='9-6-verify'></a>

#### ❓ Question: I want to guarantee that my concatenated DataFrame has no duplicate index values. How do I enforce this?

In [ ]:
# Two DataFrames with OVERLAPPING index values
part1 = pd.DataFrame({'val': [10, 20]}, index=[0, 1])
part2 = pd.DataFrame({'val': [30, 40]}, index=[1, 2])  # index 1 already in part1

# Without verification: silently creates duplicate index
silent = pd.concat([part1, part2])
print("Silent concat — duplicate index:", list(silent.index))

# With verify_integrity=True: raises ValueError on duplicates
try:
    strict = pd.concat([part1, part2], verify_integrity=True)
except ValueError as e:
    print(f"\nValueError caught: {e}")
    print("✅ Duplicate index detected before it caused problems!")

---
## 10. DataFrame.append() — Legacy Row Addition (Deprecated) <a id='10-append'></a>

#### ❓ Question: I've seen old code that uses `.append()` on a DataFrame. What was it doing, and what should I use instead?

> ⚠️ `.append()` was **deprecated in pandas 1.4** and **removed in pandas 2.0**. Always use `pd.concat()` instead.

In [ ]:
# ── OLD WAY (do NOT use — removed in pandas 2.0) ─────────────────────────────
# df_combined = df1.append(df2, ignore_index=True)  # ← DEPRECATED

# ── NEW WAY: use pd.concat() ─────────────────────────────────────────────────
df1 = employees.head(3).copy()
new_row = pd.DataFrame([{'emp_id': 99, 'name': 'Frank', 'dept_id': 20, 'salary': 95000}])

# Append a single new row
with_new_row = pd.concat([df1, new_row], ignore_index=True)
print("Added a new row with pd.concat():")
with_new_row

In [ ]:
# ── Anti-pattern: growing a DataFrame in a loop ───────────────────────────────
# BAD: this is O(n²) — avoid it!
# result = pd.DataFrame()
# for chunk in chunks:
#     result = pd.concat([result, chunk])   # ← slow, creates a new object each time

# GOOD: collect all chunks, then concat once
chunks = []
for i in range(5):
    chunk = instacart_orders.sample(3, random_state=i)
    chunks.append(chunk)          # just append to a list

result = pd.concat(chunks, ignore_index=True)   # single concat at the end
print(f"Efficiently built result with {len(result)} rows")

---
## 11. Advanced Real-World Scenarios <a id='11-advanced'></a>

### 11.1 Multi-Table Star Schema Join (Instacart) <a id='11-1-star'></a>

#### ❓ Question: I want one fully enriched DataFrame that combines order details, product names, aisle names, department names, and order metadata — all in a single table. How do I chain multiple merges?

In [ ]:
# Step 1: Start from the central fact table
star = order_products.copy()

# Step 2: Attach product names (products → fact via product_id)
star = star.merge(products,          on='product_id',   how='left')

# Step 3: Attach aisle names (aisles → fact via aisle_id)
star = star.merge(aisles,            on='aisle_id',     how='left')

# Step 4: Attach department names (departments → fact via department_id)
star = star.merge(departments,       on='department_id', how='left')

# Step 5: Attach order-level metadata (orders → fact via order_id)
star = star.merge(instacart_orders,  on='order_id',     how='left')

print(f"Final enriched DataFrame shape: {star.shape}")
print(f"Columns: {list(star.columns)}\n")
star.head(3)

In [ ]:
# Now we can answer rich business questions with a single groupby

# Q: Which department is ordered most often on Sundays (order_dow=0)?
sundays = star[star['order_dow'] == 0]
top_depts_sunday = (
    sundays.groupby('department')['order_id']
    .count()
    .sort_values(ascending=False)
    .head(5)
    .rename('order_count')
)
print("Top 5 departments on Sundays:")
top_depts_sunday

### 11.2 Self-Join — Comparing Rows Within the Same Table <a id='11-2-self-join'></a>

#### ❓ Question: I want to find all pairs of products that have been ordered together in the same order. How do I join a table to itself?

In [ ]:
# Take a small slice to keep the result manageable
small_op = order_products[order_products['order_id'].isin(
    order_products['order_id'].value_counts().head(50).index
)].copy()

# Self-join: same table on both sides, linked by order_id
self_joined = pd.merge(
    left     = small_op[['order_id', 'product_id']],
    right    = small_op[['order_id', 'product_id']],
    on       = 'order_id',
    suffixes = ('_A', '_B')   # distinguish the two product_id columns
)

# Remove rows where a product is paired with itself
pairs = self_joined[self_joined['product_id_A'] < self_joined['product_id_B']]

# Count co-occurrence frequency
top_pairs = (
    pairs.groupby(['product_id_A', 'product_id_B'])
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename('co_orders')
)
print("Top 5 most frequently co-ordered product pairs:")
top_pairs

### 11.3 Merging with Aggregated Data <a id='11-3-agg-merge'></a>

#### ❓ Question: I want to add a column showing each product's total order count alongside each individual order row. How do I merge aggregated results back onto the original table?

In [ ]:
# Step 1: compute aggregation (total orders per product)
product_popularity = (
    order_products.groupby('product_id')['order_id']
    .count()
    .reset_index()
    .rename(columns={'order_id': 'total_orders'})
)

# Step 2: merge the aggregated column back onto the product table
products_enriched = pd.merge(
    left  = products,
    right = product_popularity,
    on    = 'product_id',
    how   = 'left'          # keep all products, even those never ordered
)

# Fill NaN for products with 0 orders
products_enriched['total_orders'] = products_enriched['total_orders'].fillna(0).astype(int)

print("Top 5 most ordered products:")
products_enriched.sort_values('total_orders', ascending=False).head(5)[
    ['product_name', 'department_id', 'total_orders']
]

### 11.4 Fuzzy/Nearest Merge — `pd.merge_asof()` <a id='11-4-merge-asof'></a>

#### ❓ Question: I have stock prices recorded every second and trade events timestamped to the millisecond. How do I match each trade to the *most recent* price available at the time of the trade?

`pd.merge_asof()` performs a **nearest-key merge** — like a left join, but instead of requiring exact key matches it matches each left row to the *closest* right row (by default, the largest right key that is ≤ the left key).

In [ ]:
# Stock price snapshots (sorted by time — REQUIRED for merge_asof)
prices = pd.DataFrame({
    'time':  pd.to_datetime(['09:30:00', '09:30:05', '09:30:10', '09:30:20']),
    'price': [100.0, 101.5, 102.0, 99.5]
})

# Trade events (also sorted)
trades = pd.DataFrame({
    'time':     pd.to_datetime(['09:30:03', '09:30:08', '09:30:15', '09:30:25']),
    'quantity': [10, 25, 5, 50]
})

# merge_asof: match each trade to the LAST price before/at trade time
result = pd.merge_asof(
    left      = trades,
    right     = prices,
    on        = 'time',          # sorted key column (must be sorted ASC)
    direction = 'backward'       # 'backward' (default), 'forward', or 'nearest'
)

print("Each trade matched to the most recent price before it:")
result

In [ ]:
# direction='nearest': match to whichever price is closest in time (before or after)
result_nearest = pd.merge_asof(
    trades, prices, on='time', direction='nearest'
)
print("direction='nearest':")
result_nearest

In [ ]:
# tolerance parameter: only match if within 6 seconds
result_tol = pd.merge_asof(
    trades, prices, on='time',
    direction = 'backward',
    tolerance = pd.Timedelta('6s')   # NaN if nearest is > 6 seconds away
)
print("With 6-second tolerance (last trade at 09:30:25 gets NaN — nearest price is 25s away):")
result_tol

### 11.5 Ordered Merge — `pd.merge_ordered()` <a id='11-5-merge-ordered'></a>

#### ❓ Question: I have two time-series DataFrames. After merging I want the result sorted by the key, and I want to forward-fill any NaN values that result from the merge. Is there a clean way to do this?

`pd.merge_ordered()` is like `pd.merge()` but:
- sorts the result by the key column(s)
- supports a `fill_method` for filling NaNs after the merge

In [ ]:
# Monthly GDP data (every month)
gdp = pd.DataFrame({
    'month': pd.period_range('2024-01', periods=6, freq='M'),
    'gdp':   [21.0, 21.2, 21.5, 21.3, 21.8, 22.0]
})

# Quarterly inflation data (every 3 months)
inflation = pd.DataFrame({
    'month':     pd.period_range('2024-01', periods=2, freq='Q'),
    'inflation': [3.1, 2.9]
})

# Convert quarterly to monthly equivalent for merging
inflation['month'] = inflation['month'].dt.to_timestamp().dt.to_period('M')

# merge_ordered: sorted result + forward-fill inflation between quarters
ordered = pd.merge_ordered(
    left        = gdp,
    right       = inflation,
    on          = 'month',
    how         = 'outer',
    fill_method = 'ffill'   # forward-fill NaN inflation values
)
print("Ordered merge with forward-fill:")
ordered

### 11.6 Chaining Multiple Merges — Pipeline Style <a id='11-6-chain'></a>

#### ❓ Question: My merge logic is getting complex. How do I write multi-step merges in a clean, readable pipeline?

Use **method chaining** with `pipe()` or simply chain `.merge()` calls — Python evaluates them left to right.

In [ ]:
# ── Pipeline approach: each merge step is clearly labelled ───────────────────
result = (
    order_products                              # start from fact table
    .merge(products,         on='product_id',   how='left')   # add product name
    .merge(aisles,           on='aisle_id',     how='left')   # add aisle name
    .merge(departments,      on='department_id', how='left')  # add dept name
    .merge(instacart_orders, on='order_id',     how='left')   # add order metadata
    [['order_id', 'product_name', 'aisle', 'department',      # select columns
      'order_dow', 'order_hour_of_day', 'reordered']]
)

print(f"Final pipeline result: {result.shape}")
result.head(4)

In [ ]:
# ── Now answer a rich business question on the fully enriched table ─────────
# Q: What is the reorder rate per department?
reorder_rate = (
    result
    .groupby('department')['reordered']
    .agg(['sum', 'count'])
    .assign(reorder_rate=lambda df: (df['sum'] / df['count']).round(3))
    .sort_values('reorder_rate', ascending=False)
    .rename(columns={'sum': 'reorders', 'count': 'total_items'})
)

print("Reorder rate by department (top 10):")
reorder_rate.head(10)

---
## 12. Performance Tips & Best Practices <a id='12-performance'></a>

In [ ]:
# ── Tip 1: Sort keys before large merges ─────────────────────────────────────
# pandas can use binary search on sorted keys → faster for large DataFrames
sorted_orders   = instacart_orders.sort_values('order_id')
sorted_products = order_products.sort_values('order_id')

# sort=False prevents re-sorting the OUTPUT (the inputs are already sorted)
fast_merge = pd.merge(sorted_products, sorted_orders, on='order_id', how='left', sort=False)
print(f"Fast merge result shape: {fast_merge.shape}")

In [ ]:
# ── Tip 2: Use categorical dtypes for string keys ────────────────────────────
# Converting string keys to categorical reduces memory and speeds up groupby/merge
products_cat = products.copy()
products_cat['product_name'] = products_cat['product_name'].astype('category')

print(f"Memory before: {products['product_name'].memory_usage(deep=True) / 1024:.1f} KB")
print(f"Memory after:  {products_cat['product_name'].memory_usage(deep=True) / 1024:.1f} KB")

In [ ]:
# ── Tip 3: Always validate cardinality before a critical merge ───────────────
# Use validate to catch silent row duplication bugs early
print("Is instacart_orders unique on order_id?",
      instacart_orders['order_id'].is_unique)

print("Is order_products unique on order_id?",
      order_products['order_id'].is_unique)

In [ ]:
# ── Tip 4: Check for key nulls before merging ────────────────────────────────
# Null keys match each other in outer joins — this can cause surprise rows!
for name, df, col in [
    ('order_products',   order_products,   'product_id'),
    ('products',         products,         'product_id'),
    ('instacart_orders', instacart_orders, 'order_id'),
]:
    nulls = df[col].isna().sum()
    print(f"{name:25s}  {col:20s}  nulls: {nulls}")

In [ ]:
# ── Tip 5: Drop redundant key columns after left_on/right_on merges ──────────
# When left_on != right_on, BOTH columns appear in the result
merged = pd.merge(
    order_products.head(5),
    instacart_orders,
    left_on  = 'order_id',
    right_on = 'order_id',   # same name here, but imagine they differed
    how      = 'left'
)
print("Columns after merge:", list(merged.columns))

---
## 13. Quick Reference Cheat Sheet <a id='13-cheatsheet'></a>

In [ ]:
cheat_sheet = pd.DataFrame([
    # Function, Scenario, Key Parameters, Notes
    ['pd.merge()', 'General-purpose merge', 'how, on, left_on, right_on, suffixes, indicator, validate', 'Most powerful; all join types'],
    ['df.join()',  'Index-based join',       'on, how, lsuffix, rsuffix',                                  'Default how=left; faster syntax for index joins'],
    ['pd.concat()','Stack rows or columns',  'axis, join, ignore_index, keys, verify_integrity',           'No key matching; same-structure DataFrames'],
    ['pd.merge_asof()', 'Nearest-key merge', 'on, direction, tolerance, by',                              'Time-series; sorted keys required'],
    ['pd.merge_ordered()', 'Sorted + fill merge', 'on, how, fill_method',                                 'Time-series; output sorted by key'],
], columns=['Function', 'Use When', 'Key Parameters', 'Notes'])

display(cheat_sheet)

print("\n─── Join Types ───────────────────────────────────")
join_types = pd.DataFrame([
    ['inner', 'Keep matched rows only',           'Drop unmatched from both sides'],
    ['left',  'Keep all left rows',               'NaN in right cols for unmatched left rows'],
    ['right', 'Keep all right rows',              'NaN in left cols for unmatched right rows'],
    ['outer', 'Keep all rows from both sides',    'NaN wherever no match exists'],
    ['cross', 'Every left × every right row',     'No key needed; m×n rows'],
], columns=['how=', 'Keeps', 'NaN behaviour'])
display(join_types)

print("\n─── Key Selection Parameters ─────────────────────")
key_params = pd.DataFrame([
    ['on',           'Column name(s) shared by both tables',            'on="id"  or  on=["a","b"]'],
    ['left_on',      'Column(s) in left table when names differ',       'left_on="order_id"'],
    ['right_on',     'Column(s) in right table when names differ',      'right_on="ord_id"'],
    ['left_index',   'Use left DataFrame index as the key',             'left_index=True'],
    ['right_index',  'Use right DataFrame index as the key',            'right_index=True'],
    ['suffixes',     'Rename overlapping non-key columns',              'suffixes=("_left","_right")'],
    ['indicator',    'Add column showing match source',                 'indicator=True → _merge col'],
    ['validate',     'Assert cardinality of the merge',                 '"1:1" "1:m" "m:1" "m:m"'],
], columns=['Parameter', 'Purpose', 'Example'])
display(key_params)